# Chapter 3: Retrieval-Augmented Generation Foundations

Hands-On: A Fifteen-Line RAG Agent, Adapted from a Real Project

Extracted from: chapter_03_rag_foundations.md
Source book: Agentic AI: Building AI Agents and Retrieval Systems,
a Masterclass in LLM Agents, RAG, and Production Deployment.

Every block below was verified by direct execution before being
written into the handbook; run this file top to bottom, or copy
out the section you need. Where a step needs an API key
(OPENAI_API_KEY / ANTHROPIC_API_KEY), it is loaded from a local
.env file via python-dotenv, following Chapter 5's own security
discipline, never hardcoded.

## Installation

Run this once per environment before the cells below.

In [1]:
%pip install -q phidata lancedb duckduckgo-search python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()  # picks up OPENAI_API_KEY / ANTHROPIC_API_KEY / COHERE_API_KEY from the project's .env


True

**Step 1: Point the knowledge base at a real document and choose a vector store.**

In [3]:
from phi.agent import Agent
from phi.model.openai import OpenAIChat
from phi.knowledge.pdf import PDFUrlKnowledgeBase
from phi.vectordb.lancedb import LanceDb, SearchType
from phi.tools.duckduckgo import DuckDuckGo

db_uri = "tmp/lancedb"
knowledge_base = PDFUrlKnowledgeBase(
    # Purdue Extension's public "Ingredient Substitutions" fact sheet -- a real,
    # stable document standing in for "your-own-document-url.pdf", so the
    # question in Step 4 has real content to retrieve against.
    urls=["https://www.extension.purdue.edu/extmedia/HHS/HHS-784-W.pdf"],
    vector_db=LanceDb(table_name="my_docs", uri=db_uri, search_type=SearchType.vector),
)

**Step 2: Run ingestion once, then comment it out.**

In [4]:
knowledge_base.load(upsert=True)
# Comment this line out after the first successful run;
# re-running it re-embeds and re-inserts every chunk unnecessarily.

INFO     Creating collection

INFO     Loading knowledge base

INFO     Reading: https://www.extension.purdue.edu/extmedia/HHS/HHS-784-W.pdf

INFO     Added 12 documents to knowledge base

**Step 3: Wire the retriever and a web-search fallback into one agent.**

In [5]:
rag_agent = Agent(
    model=OpenAIChat(id="gpt-4o"),
    agent_id="rag-agent",
    knowledge=knowledge_base,
    tools=[DuckDuckGo()],
    show_tool_calls=True,
    markdown=True,
)

**Step 4: Ask a question and inspect which source the agent actually used.**

In [6]:
response = rag_agent.run("What does the document say about ingredient substitutions?")
print(response.content)


Running:
 - search_knowledge_base(query=ingredient substitutions)

The document provides detailed information on ingredient substitutions, organized in a chart format. It includes various alternative options that can be used in place of certain ingredients when preparing a dish. Here are some key points:

- **Purpose**: Ingredient substitution is useful when you're missing an ingredient and prefer not to make a store trip. However, it's meant for unexpected situations and should be done considering how substitutes might alter the taste, color, moisture content, or texture of the product.
- **Considerations**: Each ingredient has a specific function in a recipe, so substituting one for another might change the final product.
- **Example Substitutes**:
  - Orange can be substituted with orange peel, dried or fresh, or orange extract.
  - Pumpkin pie spice can be substituted with a mix of cinnamon, ginger, allspice, and nutmeg.
  - Baking powder alternatives include various combinations 